# Effect of classification model

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.ticker import ScalarFormatter

csv_dir = Path("../../benchmarks/v1/mean_profiles/")
csvs = sorted(csv_dir.glob("shape_features.minirocket.*.csv"))

X = pd.read_csv("../../benchmarks/v1/dimless.csv")
idxs = np.load("../../benchmarks/v1/index.npy")

In [ ]:
slurries = ["G50", "G45", "G40", "G40+IPA"]

Cas = X.iloc[idxs]["capillary_number"].unique()
Cas_sorted = np.sort(Cas)
cmap = plt.get_cmap("viridis", len(Cas_sorted))
norm = mcolors.BoundaryNorm(
    np.concatenate(
        [
            [Cas_sorted[0] * 0.9],
            (Cas_sorted[:-1] + Cas_sorted[1:]) / 2,
            [Cas_sorted[-1] * 1.1],
        ]
    ),
    ncolors=len(Cas_sorted),
)

In [ ]:
TARGET = "phi"

for path in csvs:
    data = pd.concat([X, pd.read_csv(path)], axis=1).iloc[idxs]

    fig, axes = plt.subplots(
        1, len(slurries), sharex=True, sharey="row", figsize=(10, 5)
    )

    for slurry, ax in zip(slurries, axes):
        ok = data["slurry"] == slurry
        subdata = data[ok]

        for ca in subdata["capillary_number"].unique():
            ok = subdata["capillary_number"] == ca

            ax.scatter(
                subdata[ok]["gap_to_thickness_ratio"],
                subdata[ok][TARGET],
                color=cmap(norm(ca)),
            )

    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cbar = fig.colorbar(
        sm, ax=axes, orientation="horizontal", location="top", pad=0.2, aspect=30
    )
    cbar.set_label("Ca")
    quartile_vals = np.quantile(Cas_sorted, [0, 0.25, 0.5, 0.75, 1.0])
    nearest_cas = [Cas_sorted[np.argmin(np.abs(Cas_sorted - q))] for q in quartile_vals]
    cbar.set_ticks([round(ca, 3) for ca in nearest_cas])
    cbar.ax.xaxis.set_major_formatter(ScalarFormatter())

    fig.suptitle(path.stem.split(".")[-1])
    fig.supxlabel("Rgt")
    fig.supylabel(TARGET)
    fig.show()